# 01 — Getting started

This notebook walks through the `lammps` Python API — the same API as the
[official LAMMPS Python module](https://docs.lammps.org/Python_module.html),
driving the lammps.js WebAssembly engine. Install the bundled package first
(it resolves from the wheel shipped with this site, not from PyPI):

In [ ]:
%pip install lammps-js

## Create a LAMMPS instance

One browser-specific difference from the official module: the WebAssembly
engine loads asynchronously, so creation takes a single `await`. Everything
after that is synchronous, just like native LAMMPS.

In [ ]:
from lammps import lammps

lmp = await lammps()   # prints the LAMMPS banner
print("LAMMPS version:", lmp.version())

## Build and run a system

`commands_string` runs a block of LAMMPS input (there is also `command` for a
single command, `commands_list` for a list, and `file` for an input file —
see the [next notebook](02-scripts-and-files.ipynb)). This creates a small
Lennard-Jones fcc crystal with enough kinetic energy to melt:

In [ ]:
lmp.commands_string("""
units         lj
atom_style    atomic
lattice       fcc 0.8442
region        box block 0 3 0 3 0 3
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 3.0 87287
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
thermo        100
""")
lmp.command("run 300")

## Query the state

The query API is the official one: `get_natoms`, `get_thermo` (any thermo
keyword), `extract_global`, `extract_box`:

In [ ]:
print("atoms:      ", lmp.get_natoms())
print("timestep:   ", lmp.extract_global("ntimestep"))
print("temperature:", lmp.get_thermo("temp"))
print("pot. energy:", lmp.get_thermo("pe"))
print("pressure:   ", lmp.get_thermo("press"))

boxlo, boxhi, xy, yz, xz, periodicity, box_change = lmp.extract_box()
print("box:        ", boxlo, "to", boxhi)

## Per-atom data as numpy arrays

`extract_atom` returns **numpy arrays** (copies — in the official module you
get raw C pointers instead). Positions are `(natoms, 3)`:

In [ ]:
x = lmp.extract_atom("x")
ids = lmp.extract_atom("id")
types = lmp.extract_atom("type")

print("x.shape:", x.shape, " dtype:", x.dtype)
print("center of mass:", x.mean(axis=0))

Any other per-atom quantity is available through atom-style variables. For
example the x-velocities (the official module would use
`lmp.extract_atom("v")` — the wasm bindings expose per-atom data through the
variable mechanism instead):

In [ ]:
from lammps import LMP_VAR_ATOM

lmp.command("variable vx atom vx")
vx = lmp.extract_variable("vx", vartype=LMP_VAR_ATOM)
print("vx.shape:", vx.shape, " mean:", vx.mean(), " std:", vx.std())

## The session is live

State persists between cells — run some more steps and look again. When you
are done, `close()` shuts the instance down:

In [ ]:
lmp.command("run 200")
print("temperature now:", lmp.get_thermo("temp"))
lmp.close()

## What's different from the official module?

- `lmp = await lammps(...)` instead of `lammps(...)` — wasm loads async.
- `extract_atom` / `extract_compute` / `extract_fix` / `extract_variable`
  return **numpy copies**, not ctypes pointers (`lmp.numpy.…` works too).
- `extract_atom` exposes `"x"`, `"id"`, `"type"`; use atom-style variables
  for everything else.
- LAMMPS runs in an in-memory filesystem: `lmp.file(path)` copies a local
  notebook file in transparently, or takes the body via `contents=`.

Next: [02 — Scripts and files](02-scripts-and-files.ipynb).